In [ ]:
import numpy as np
import matplotlib.pylab as plt
import matplotlib as mpl
mpl.rcParams['font.size'] = 16

In [ ]:
def f_pres(rho,eint):
    return ...

def f_eint(rho,pres):
    return ...

def f_cs(pres,rho):
    return ...
    
def initial_data(x,IC):

    nmax = len(x)
    rho = np.zeros(nmax)
    pres = np.zeros(nmax)
    v = np.zeros(nmax)
    ener = np.zeros(nmax)
    cs = np.zeros(nmax)
    if IC=='sod':
        #create sod initial data
        rho = ...
        pres = ...
        ener = ...
        cs = ...
    elif IC=='custom':
        #create custom initial data
        rho = ...
        pres = ...
        ener = ...
        cs = ...
    else:
        print("Must define initial data")

    return rho,pres,v,ener,cs

def set_boundaries(rho,pres,v,ener,cs,nguard):

    nmax = len(rho)

    #reflecting boundary conditions
    for i in range(nguard):
        rho[i] = ... #rho[2*ngard-1-i]
        pres[i] = ... #pres[2*...]
        v[i] = ... #-v[2*...]
        ener[i] = ...
        cs[i] = ...

        rho[nmax-i-1] = ... #rho[nmax - 2*nguard+1]
        pres[nmax-i-1] = ... #pres[nmax - 2*nguard+1]
        v[nmax-i-1] = ...   #-v[...]
        ener[nmax-i-1] = ...
        cs[nmax-i-1] = ...


def reconstruct(var,nguard):
    nmax = len(var)
    varp = np.zeros(nmax)
    varm = np.zeros(nmax)
    
    for m in range(nguard-1,nmax-nguard+1):
        #s1 = var[m] - var[m-1]
        #s2 = var[m+1] - var[m]
        #if s1*s2<0.0:
            #delta = 0.0
        #else:
            #delta = np.sign(s1)*min(np.abs(s1), np.abs(s2))
        ...
        
        varp[m] = ... #varp[m] = var[m] + 0.5 * delta
        varm[m] = ... #varp[m] = var[m] - 0.5 * delta

    return varp,varm
    

def get_flux(x,rho,pres,v,ener,cs,nguard):

    nmax = len(rho)
    dx = x[1]-x[0]

    #get the cell average internal energy
    eint = f_eint(rho,pres)
    
    #reconstruct vars to interfaces    
    rhop,rhom = reconstruct(rho,nguard)
    vp,vm = reconstruct(v,nguard)
    presp,presm = reconstruct(pres,nguard) 

    #calculate new hydro variables at interface
    eintp = ... #f_eint(rhop, eintp)
    eintm = ... #f_eint(rhom, eintm)
    csp = ...   #f_cs(presp, rhop)
    csm = ...   #f_cs(presp, rhom)
    Ep = ...    #rhop*(0.5*vp)
    Em = ...

    #calculate flux vector on the interfaces
    flux_rho_p = ...
    flux_rho_m = ...

    flux_momx_p = ...
    flux_momx_m = ...

    flux_ener_p = ...
    flux_ener_m = ...

    #now combine to get flux at interface m+1/2 via the Riemann solver.  here L is m+ and R is (m+1)-
    flux_rho = np.zeros(nmax)
    flux_momx = np.zeros(nmax)
    flux_ener = np.zeros(nmax)
    for m in range(nguard-1,nmax-nguard):
        #also need the characteristic speeds
        maxlambda = ...
        minlambda = ...
    
        flux_rho[m] = ...
        flux_momx[m] = ...
        flux_ener[m] = ...

    return flux_rho,flux_momx,flux_ener


def evolve_euler_problem(nzones=50,xmin=0.0,xmax=1.0,C=0.25,tend=0.2,IC='sod',savefig=False):
    #construct grid, dx is the grid spacing, set by the xmin and xmax and the number of zones
    dx = (xmax-xmin)/nzones
    #use two guard zones for boundary conditions
    nguard = 2
    
    #set the x array, we want the x value to be the zone center coordinate, so shift the array from linspace by dx/2
    x = np.linspace(xmin-nguard*dx,xmax+nguard*dx,nzones+2*nguard,endpoint=False)+dx/2

    #set initial conditions
    rho,pres,v,ener,cs = initial_data(x,IC)
    set_boundaries(rho,pres,v,ener,cs,nguard)

    #make a copy of ID to return
    rho0 = rho.copy()
    pres0 = pres.copy()
    v0 = v.copy()

    time=0.0
    count=0
    figcount=0
    while time<tend:
        rhon = rho.copy()
        momx = rho*v
        momxn = momx.copy()
        enern = ener.copy()
        
        dt = C*dx/np.max(cs+np.abs(v))
        #prevect time from exceeding tend
        if time+dt>tend:
            dt = tend-time


        #RK step 1
        ldt = ..

        #boundary conditions
        set_boundaries(rho,pres,v,ener,cs,nguard)
        
        #get the flux at the interfaces
        flux_rho,flux_momx,flux_ener = get_flux(x,rho,pres,v,ener,cs,nguard)
        
        #advance the density field forward in time by dt
        rho[1:] = ...
        momx[1:] = ...
        ener[1:] = ...

        #recover the primatives
        v = ...
        eint = ...
        pres = ...
        cs = ...

        #RK step 2??
        
        set_boundaries(rho,pres,v,ener,cs,nguard)

        if count%10==0 and savefig==True: 
            plt.plot(x[nguard:-nguard],rho[nguard:-nguard],label='rho')
            plt.plot(x[nguard:-nguard],pres[nguard:-nguard],label='pres')
            plt.plot(x[nguard:-nguard],v[nguard:-nguard],label='v')
            plt.xlabel('x')
            timestring = str(int((time+dt)*10000)/10000.)
            plt.title("time="+timestring)
            plt.ylabel("rho, pres, v")
            plt.legend(frameon=False)
            plt.savefig("sod_"+str(figcount).zfill(3)+".png",bbox_inches='tight')
            plt.clf()
            figcount+= 1
        count += 1

        #advance the time variable, and loop
        time = time + dt

    return x,rho,v,pres

In [ ]:
#for i in [50,100,200,400,800,1600]:
gamma=1.4
finaldata={}
for i in [50,100,200,400]:
    x,rho,v,pres = evolve_euler_problem(nzones=i,xmin=0.0,xmax=1.0,C=0.25,tend=0.2,IC='sod')
    rho0,pres0,v0,ener0,cs0 = initial_data(x,'sod')
    finaldata[i] = {"x":x,"rho":rho,"pres":pres,"v":v}

In [ ]:
nguard=2
xe,rhox,velx,presx = np.loadtxt("sod_exact.txt",skiprows=2,unpack=True)
plt.plot(xe,rhox,'k')
for set in finaldata:
    plt.plot(finaldata[set]['x'][nguard:-nguard],finaldata[set]['rho'][nguard:-nguard],label=set)
plt.legend(frameon=False,fontsize=12)
plt.xlabel("x")
plt.ylabel("Density")
plt.savefig("sod_dens.pdf",bbox_inches='tight')
plt.show()

plt.plot(xe,velx,'k')
for set in finaldata:
    plt.plot(finaldata[set]['x'][nguard:-nguard],finaldata[set]['v'][nguard:-nguard],label=set)
plt.legend(frameon=False,fontsize=12)
plt.xlabel("x")
plt.ylabel("Velocity")
plt.savefig("sod_velx.pdf",bbox_inches='tight')
plt.show()

plt.plot(xe,presx,'k')
for set in finaldata:
    plt.plot(finaldata[set]['x'][nguard:-nguard],finaldata[set]['pres'][nguard:-nguard],label=set)
plt.legend(frameon=False,fontsize=12)
plt.xlabel("x")
plt.ylabel("Pressure")
plt.savefig("sod_pres.pdf",bbox_inches='tight')
plt.show()

In [ ]:
nguard=2
xe,rhox,velx,presx = np.loadtxt("sod_exact.txt",skiprows=2,unpack=True)
plt.plot(xe,rhox*velx,'k')
for set in finaldata:
    plt.plot(finaldata[set]['x'][nguard:-nguard],finaldata[set]['rho'][nguard:-nguard]*finaldata[set]['v'][nguard:-nguard],label=set)
plt.legend(frameon=False,fontsize=12)
plt.xlabel("x")
plt.ylabel("Density flux")
plt.savefig("sod_dens_flux.pdf",bbox_inches='tight')
plt.show()

plt.plot(xe,rhox*velx**2+presx,'k')
for set in finaldata:
    plt.plot(finaldata[set]['x'][nguard:-nguard],finaldata[set]['rho'][nguard:-nguard]*finaldata[set]['v'][nguard:-nguard]**2+finaldata[set]['pres'][nguard:-nguard],label=set)
plt.legend(frameon=False,fontsize=12)
plt.xlabel("x")
plt.ylabel("Momentum flux")
plt.savefig("sod_momx_flux.pdf",bbox_inches='tight')
plt.show()
eintx = presx/((gamma-1)*rhox)
plt.plot(xe,(rhox*eintx+rhox*0.5*velx**2+presx)*velx,'k')
for set in finaldata:
    eintf = finaldata[set]['pres']/((gamma-1)*finaldata[set]['rho'])
    ef = (finaldata[set]['rho']*eintf+finaldata[set]['rho']*0.5*finaldata[set]['v']**2+finaldata[set]['pres'])*finaldata[set]['v']
    plt.plot(finaldata[set]['x'][nguard:-nguard],ef[nguard:-nguard],label=set)
plt.legend(frameon=False,fontsize=12)
plt.xlabel("x")
plt.ylabel("Energy Flux")
plt.savefig("sod_energy_flux.pdf",bbox_inches='tight')
plt.show()

In [ ]:
rho0,pres0,v0,ener0,cs0 = initial_data(x,'sod')

plt.plot(x,rho0,'k:')
plt.plot(xe,rhox,'k')
plt.xlabel("x")
plt.ylabel("Density")
plt.show()